# Comparing VaR Methods: Historical, Parametric, Monte Carlo

In this notebook we compare three different approaches to estimating Value-at-Risk (VaR):

1. Historical Simulation
2. Parametric (Gaussian) VaR
3. Monte Carlo Simulation

We will:
- Build a simple 4-asset portfolio
- Compute 95% 5-day VaR using all three methods
- Compare results in a table and a visualization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from src.var_methods import historical_var, parametric_var, monte_carlo_var

## Load Data
We use four assets: SPY, BND, GLD, QQQ

In [ ]:
tickers = ["SPY", "BND", "GLD", "QQQ"]
data = yf.download(tickers, start="2020-01-01", end="2025-01-01")["Close"]
log_returns = np.log(data / data.shift(1)).dropna()

log_returns.head()

## Compute VaR Estimates
- Horizon: 5 days
- Confidence: 95%
- Portfolio: Equal weights, $1,000,000 value

In [ ]:
weights = np.array([0.25, 0.25, 0.25, 0.25])
portfolio_value = 1_000_000
alpha = 0.05
horizon = 5

hist_var = historical_var(log_returns, weights, alpha=alpha, horizon=horizon, portfolio_value=portfolio_value)
param_var = parametric_var(log_returns, weights, alpha=alpha, horizon=horizon, portfolio_value=portfolio_value)
mc_var = monte_carlo_var(log_returns, weights, alpha=alpha, horizon=horizon, sims=10000, portfolio_value=portfolio_value, seed=42)

results = pd.DataFrame({
    "Method": ["Historical", "Parametric (Gaussian)", "Monte Carlo"],
    "VaR (95%, 5-day)": [hist_var, param_var, mc_var]
})
results

## Visual Comparison
We plot the VaR estimates side by side.

In [ ]:
plt.figure(figsize=(8,6))
plt.bar(results["Method"], results["VaR (95%, 5-day)"], color=["blue", "green", "orange"])
plt.ylabel("VaR ($)")
plt.title("Comparison of VaR Estimates (95%, 5-day horizon)")
plt.show()

## Summary
- Historical VaR: purely data-driven, no assumptions.
- Parametric VaR: quick, but assumes Normality.
- Monte Carlo VaR: flexible, incorporates covariance, but computationally heavier.

The differences highlight why risk managers often use **multiple methods** together for robustness.